# Day 3 · DIY-2 — Data model + one-retry self-heal

**Goal:** make a bad model response *non-fatal*. You'll define a data model with one validation rule, then wrap the call so a validation failure feeds the error back and retries **once** — then fails safe.

**Done when:** a deliberately bad value triggers exactly one retry, then a valid record *or* a clean caught error. Never an unhandled crash.

> Runs in Google Colab. Set your `OPENAI_API_KEY` in the first cell. (Swap to Claude/Gemini later — the pattern is identical.)


In [ ]:
!pip -q install openai pydantic
import os, json
from openai import OpenAI
from pydantic import BaseModel, field_validator, ValidationError

# Colab: os.environ['OPENAI_API_KEY'] = 'sk-...'  # or use Colab secrets
client = OpenAI()
MODEL = 'gpt-4o-2024-08-06'   # pin a snapshot that supports Structured Outputs


## 1 · Define the contract as a data model
Fill the `TODO`: add a validator that rejects any priority outside `low / med / high`.


In [ ]:
class Ticket(BaseModel):
    customer: str
    priority: str
    summary: str

    @field_validator('priority')
    @classmethod
    def check_priority(cls, v):
        allowed = {'low', 'med', 'high'}
        if v not in allowed:
            raise ValueError(f"priority must be one of {sorted(allowed)}, got {v!r}")
        return v


## 2 · The self-heal wrapper
Fill the `TODO`s: on `ValidationError`, feed the error text back and retry **once** (cap = 1).


In [ ]:
def extract_ticket(text, max_retries=1):
    msgs = [
        {'role': 'system', 'content': (
            'Extract a support ticket as JSON with keys customer, priority, summary. '
            'priority must be exactly one of: low, med, high.'
        )},
        {'role': 'user', 'content': text},
    ]
    for attempt in range(max_retries + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=msgs, response_format={'type': 'json_object'},
        )
        raw_text = resp.choices[0].message.content
        try:
            return Ticket.model_validate_json(raw_text)   # valid, typed Ticket
        except ValidationError as e:
            if attempt == max_retries:
                return None                                 # fail safe -> hand off to a human
            msgs.append({'role': 'assistant', 'content': raw_text})
            msgs.append({'role': 'user', 'content': f'That was invalid: {e}. Please fix and resend as JSON.'})
    return None

## 3 · Test with a deliberately hard input
The phrase *“this is urgent!!”* tempts the model toward `priority='urgent'` — which your validator must reject, triggering the retry.

> **Demo note:** `gpt-4o-2024-08-06` is very compliant with the system prompt and will usually self-correct to `'high'` on the first try — so a live call may not actually fail. Run the cell below for the real (unscripted) attempt, then run the **Demo Reliability Toggle** cell after it if you want to *guarantee* the fail → retry → heal sequence is visible on stage (same idea as the deck's Self-Heal Simulator).

In [ ]:
bad = 'Customer Meera says the app keeps crashing on checkout. This is urgent!!'
result = extract_ticket(bad)
print(result)


## 3b · Demo Reliability Toggle (optional, for live sessions)
Simulates one bad response (`priority='urgent'`) so the fail → retry → heal sequence is guaranteed to show, instead of depending on the model happening to misbehave live.

In [ ]:
from unittest.mock import patch
from types import SimpleNamespace

def demo_forced_failure(text):
    """DEMO AID ONLY: forces attempt 1 to return an invalid priority so the
    retry/self-heal path is visible live, then lets attempt 2 hit the real API."""
    bad_json = '{"customer": "Meera", "priority": "urgent", "summary": "App crashes on checkout."}'
    real_create = client.chat.completions.create
    state = {'n': 0}
    def fake_create(*a, **kw):
        state['n'] += 1
        if state['n'] == 1:
            return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=bad_json))])
        return real_create(*a, **kw)
    with patch.object(client.chat.completions, 'create', side_effect=fake_create):
        return extract_ticket(text)

demo_result = demo_forced_failure(bad)
print(demo_result)

### ✅ Acceptance
- One bad value → **exactly one** retry → valid `Ticket` or a clean `None`/handoff.
- No unhandled exception escapes `extract_ticket`.

**Stretch:** add a `field_validator` that title-cases `customer`, and log how many retries each call used.
